In [1]:
# importing

import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt

In [ ]:
def phase_fold(t, period):
    """Compute the phase of each time point given a period."""
    return np.mod(t, period) / period

def log_likelihood(params, t, flux, flux_err):
    """Compute the log-likelihood of the data given the model parameters."""
    period, phase_offset, amplitude = params
    phase = phase_fold(t, period)
    model = amplitude * np.sin(2 * np.pi * (phase + phase_offset))
    ll = -0.5 * np.sum(((flux - model) / flux_err) ** 2 + np.log(2 * np.pi * flux_err ** 2))
    return -ll  # return negative because we minimize

def compute_odds_ratio(t, flux, flux_err, period_range, n_trials=100):
    """Compute the odds ratio over a range of periods."""
    no_signal_ll = -0.5 * np.sum((flux / flux_err) ** 2 + np.log(2 * np.pi * flux_err ** 2))
    max_logL_signal = -np.inf
    
    for trial in range(n_trials):
        initial_guess = [
            np.random.uniform(*period_range),  # random period in the given range
            np.random.uniform(0, 1),           # random phase offset
            np.random.uniform(0, np.ptp(flux)) # random amplitude guess
        ]
        result = minimize(log_likelihood, initial_guess, args=(t, flux, flux_err), bounds=[period_range, (0, 1), (0, np.inf)])
        max_logL_signal = max(max_logL_signal, -result.fun)
    
    odds_ratio = np.exp(max_logL_signal - no_signal_ll)
    return odds_ratio

def compute_probability_of_variability(t, flux, flux_err, period_range, n_trials=10):
    """Compute the probability of variability given a light curve."""
    odds_ratio = compute_odds_ratio(t, flux, flux_err, period_range, n_trials)
    probability_of_variability = odds_ratio / (1 + odds_ratio)
    return probability_of_variability

def classify_variability(probability_of_variability):
    """Classify the variability strength based on the probability of variability."""
    if probability_of_variability > 0.9:
        return "Strong Variability"
    elif probability_of_variability > 0.7:
        return "Moderate Variability"
    else:
        return "Weak or No Variability"

# Generate synthetic light curve data
n_points = 50
time = np.linspace(0, 50, n_points)
true_rate = 30
period = 10
amplitude = 10

# Generate synthetic light curve
def generate_synthetic_light_curve(time, true_rate, period, amplitude):
    phase = np.mod(time, period) / period
    true_counts = true_rate + amplitude * np.sin(2 * np.pi * phase)
    observed_counts = np.random.poisson(true_counts)
    uncertainties = np.sqrt(observed_counts)
    return observed_counts, uncertainties

flux, flux_err = generate_synthetic_light_curve(time, true_rate, period, amplitude)

# Compute the probability of variability and classify the light curve
period_range = (0.5, 30.0)
probability_of_variability = compute_probability_of_variability(time, flux, flux_err, period_range)
variability_classification = classify_variability(probability_of_variability)

# Print the results
print(f"Probability of Variability: {probability_of_variability:.4f}")
print(f"Variability Classification: {variability_classification}")

# Plot the synthetic light curve with error bars
plt.errorbar(time, flux, yerr=flux_err, fmt='o', ecolor='red', capsize=3, label='Synthetic X-ray Light Curve')
plt.xlabel('Time (days)')
plt.ylabel('Counts')
plt.title(f'Synthetic X-ray Light Curve\nVariability: {variability_classification}')
plt.legend()
plt.show()